# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [1]:
import pandas as pd
import numpy as np
import faiss
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
import os
import time
from dotenv import load_dotenv

# API key aku taruh di file .env (GEMINI_API_KEY=...), biar ga ke-push ke github
load_dotenv()
client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

df = pd.read_csv('../data/ecommerce_faq_knowledge_base.csv')
print('Total entri knowledge base:', len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


In [2]:
# cek struktur datanya dulu
print('Kolom:', df.columns.tolist())
print('Shape:', df.shape)
print()
print('Missing value per kolom:')
print(df.isna().sum())
print()
print('Duplikat pertanyaan:', df.duplicated('question').sum())
print()
print('Jumlah FAQ per kategori:')
print(df['category'].value_counts())

Kolom: ['faq_id', 'category', 'question', 'answer']
Shape: (59, 4)

Missing value per kolom:
faq_id      0
category    0
question    0
answer      0
dtype: int64

Duplikat pertanyaan: 0

Jumlah FAQ per kategori:
category
Pemesanan           9
Pengiriman          9
Pembayaran          8
Pengembalian        8
Akun                7
Produk              7
Promo               6
Customer Service    5
Name: count, dtype: int64


In [3]:
# liat panjang teks question & answer, buat gambaran aja seberapa panjang yang mau di-embed
df['q_len'] = df['question'].str.split().str.len()
df['a_len'] = df['answer'].str.split().str.len()
df[['q_len', 'a_len']].describe()

,q_len,a_len
count,59.000000,59.000000
mean,7.644068,17.169492
std,1.517288,2.600746
min,5.000000,11.000000
25%,6.500000,15.000000
50%,8.000000,17.000000
75%,9.000000,19.000000
max,11.000000,23.000000


Dari cek di atas: ada 59 FAQ, 4 kolom (`faq_id`, `category`, `question`, `answer`), 8 kategori (Pemesanan, Pengiriman, Pembayaran, Pengembalian, Akun, Produk, Promo, Customer Service). Datanya bersih, ga ada yang kosong atau dobel. Jawabannya juga pendek-pendek (cuma 1-3 kalimat), jadi ga perlu chunking, 1 FAQ = 1 unit retrievable udah cukup.

## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [4]:
# kategori ikut dimasukin biar konteksnya lebih kebaca sama model embedding
df['content'] = 'Kategori: ' + df['category'] + '\nQ: ' + df['question'] + '\nA: ' + df['answer']
documents = df['content'].tolist()
print('Jumlah unit retrievable:', len(documents))
print('Contoh unit retrievable:')
print(documents[0])

Jumlah unit retrievable: 59
Contoh unit retrievable:
Kategori: Pemesanan
Q: Bagaimana cara melakukan pemesanan di platform ini?
A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

In [5]:
EMBED_MODEL = 'gemini-embedding-001'  # text-embedding-004 udah ga ada di list model, jadi pakai ini

def get_embedding(text, task_type='RETRIEVAL_QUERY'):
    result = client.models.embed_content(
        model=EMBED_MODEL,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    return result.embeddings[0].values

def get_embeddings_batch(texts, task_type='RETRIEVAL_DOCUMENT', batch_size=50):
    # dibikin per batch biar ga manggil API 59x satu-satu
    all_emb = []
    for i in range(0, len(texts), batch_size):
        result = client.models.embed_content(
            model=EMBED_MODEL,
            contents=texts[i:i + batch_size],
            config=types.EmbedContentConfig(task_type=task_type)
        )
        all_emb.extend([e.values for e in result.embeddings])
    return all_emb

embeddings = get_embeddings_batch(documents)
embeddings = np.array(embeddings).astype('float32')
print('Shape embeddings:', embeddings.shape)

Shape embeddings: (59, 3072)


In [6]:
# verifikasi sebelum masuk FAISS: jumlah baris harus = jumlah FAQ
assert embeddings.shape[0] == len(df), 'jumlah embedding ga sama dengan jumlah FAQ!'
print('Dimensi tiap vektor:', embeddings.shape[1])
print('Ada NaN?', np.isnan(embeddings).any())

# normalisasi biar jarak L2 nya sebanding sama cosine similarity
faiss.normalize_L2(embeddings)
print('Norm 5 vektor pertama:', np.linalg.norm(embeddings[:5], axis=1).round(4))

Dimensi tiap vektor: 3072
Ada NaN? False
Norm 5 vektor pertama: [1. 1. 1. 1. 1.]


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [7]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)
print('Sesuai jumlah FAQ?', index.ntotal == len(df))

# simpan index ke file biar bisa dipakai lagi tanpa embed ulang
os.makedirs('../vector_store', exist_ok=True)
faiss.write_index(index, '../vector_store/faq_index.faiss')
df[['faq_id', 'category', 'question', 'answer']].to_csv('../vector_store/faq_metadata.csv', index=False)
print('Index disimpan ke ../vector_store/faq_index.faiss')

Jumlah vektor tersimpan di index: 59
Sesuai jumlah FAQ? True
Index disimpan ke ../vector_store/faq_index.faiss


In [8]:
# cek index yang disimpan bisa di-load lagi
index_loaded = faiss.read_index('../vector_store/faq_index.faiss')
print('Jumlah vektor di index hasil load:', index_loaded.ntotal)

Jumlah vektor di index hasil load: 59


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [9]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_embedding(query)]).astype('float32')
    faiss.normalize_L2(query_embedding)
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        dist = float(distances[0][rank])
        results.append({
            'rank': rank + 1,
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': round(dist, 4),
            # karena vektornya udah dinormalisasi: L2^2 = 2 - 2*cos
            'similarity': round(1 - dist / 2, 4)
        })
    return results

def show_results(results):
    return pd.DataFrame(results)[['rank', 'category', 'question', 'distance', 'similarity']]

# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
show_results(test_results)

,rank,category,question,distance,similarity
0,1,Pengiriman,Bagaimana jika paket saya belum sampai melebih...,0.4028,0.7986
1,2,Pengiriman,Kenapa status pengiriman saya tidak berubah se...,0.4353,0.7823
2,3,Pengiriman,Kenapa nomor resi saya belum bisa dilacak di w...,0.4690,0.7655


In [10]:
# coba satu lagi pakai bahasa yang lebih santai
show_results(semantic_search('gimana caranya pake voucher diskon?'))

,rank,category,question,distance,similarity
0,1,Promo,Bagaimana cara menggunakan kode voucher saat c...,0.4398,0.7801
1,2,Promo,Apakah member baru mendapat diskon khusus?,0.5428,0.7286
2,3,Promo,Kenapa kode voucher saya tidak bisa digunakan?,0.5741,0.7129


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [11]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"[FAQ {r['rank']}] (Kategori: {r['category']}, similarity: {r['similarity']})\nPertanyaan: {r['question']}\nJawaban: {r['answer']}"
        for r in search_results
    )
    prompt = f'''### PERSONA
Kamu adalah asisten customer service e-commerce yang ramah, sopan, dan membantu. Kamu bicara pakai bahasa Indonesia yang santai tapi tetap sopan.

### TASK
Jawab pertanyaan pelanggan HANYA berdasarkan informasi di bagian CONTEXT.
Aturan:
- Jangan pakai pengetahuan umum di luar CONTEXT dan jangan mengarang info (angka, biaya, durasi, nama layanan, dll).
- FAQ di CONTEXT adalah hasil pencarian otomatis, jadi belum tentu semuanya relevan. Pakai yang memang nyambung sama pertanyaan saja.
- Kalau informasinya tidak ada di CONTEXT, bilang terus terang bahwa kamu belum punya informasi tersebut, lalu sarankan pelanggan menghubungi customer service.

### CONTEXT
{context}

### FORMAT
- Jawab langsung ke intinya dalam 2-4 kalimat.
- Kalau ada langkah-langkah, boleh pakai poin singkat.
- Jangan menyebut kata "CONTEXT" atau "FAQ" di jawaban.

### PERTANYAAN PELANGGAN
{query}

### JAWABAN'''
    return prompt

# liat dulu prompt jadinya kayak apa
print(build_prompt('kenapa paket saya belum sampai?', test_results))

### PERSONA
Kamu adalah asisten customer service e-commerce yang ramah, sopan, dan membantu. Kamu bicara pakai bahasa Indonesia yang santai tapi tetap sopan.

### TASK
Jawab pertanyaan pelanggan HANYA berdasarkan informasi di bagian CONTEXT.
Aturan:
- Jangan pakai pengetahuan umum di luar CONTEXT dan jangan mengarang info (angka, biaya, durasi, nama layanan, dll).
- FAQ di CONTEXT adalah hasil pencarian otomatis, jadi belum tentu semuanya relevan. Pakai yang memang nyambung sama pertanyaan saja.
- Kalau informasinya tidak ada di CONTEXT, bilang terus terang bahwa kamu belum punya informasi tersebut, lalu sarankan pelanggan menghubungi customer service.

### CONTEXT
[FAQ 1] (Kategori: Pengiriman, similarity: 0.7986)
Pertanyaan: Bagaimana jika paket saya belum sampai melebihi estimasi?
Jawaban: Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.

[FAQ 2] (Kategori: Pengiriman, similarity:

## 7. Mengintegrasikan LLM

In [12]:
LLM_MODEL = 'gemini-2.5-flash'  # gemini-2.0-flash udah ga ada di list model

def generate_answer(query, top_k=3):
    # alurnya: query -> retrieval -> prompt -> jawaban
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    # free tier cuma 5 request/menit, jadi kalau kena 429 tunggu bentar terus coba lagi
    for attempt in range(5):
        try:
            response = client.models.generate_content(
                model=LLM_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.3)
            )
            return response.text, search_results
        except Exception as e:
            if '429' in str(e) and attempt < 4:
                print('kena rate limit, tunggu 30 detik...')
                time.sleep(30)
            else:
                raise

answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print('-', s['question'], f"(similarity: {s['similarity']})")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Jawaban LLM:
Halo Kak! Tentu saja bisa. Fitur cicilan tersedia untuk transaksi menggunakan kartu kredit tertentu, dengan nominal minimal yang sudah ditentukan oleh pihak bank, ya.

Sumber FAQ yang digunakan:
- Apakah saya bisa membayar dengan cicilan? (similarity: 0.814)
- Metode pembayaran apa saja yang tersedia? (similarity: 0.6978)
- Apakah bisa membayar menggunakan lebih dari satu metode sekaligus? (similarity: 0.6916)


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [13]:
test_queries = [
    # 1-2: jelas ada di FAQ
    ('Ada di FAQ', 'Bagaimana cara membatalkan pesanan?'),
    ('Ada di FAQ', 'Berapa lama proses refund setelah barang dikembalikan?'),
    # 3-5: parafrase / sinonim, katanya beda dari FAQ asli
    ('Parafrase', 'duit udah kepotong tapi orderan masih dibilang belum lunas, gimana nih?'),
    ('Parafrase', 'aku ga inget kata sandi buat masuk, caranya reset gimana ya'),
    ('Parafrase', 'barang yang dateng beda sama yang aku order'),
    # 6: ga ada di FAQ
    ('Tidak ada di FAQ', 'Apakah bisa kirim barang ke luar negeri, misalnya ke Jepang?'),
]

test_log = []
for jenis, q in test_queries:
    print(f'[{jenis}] Query:', q)
    answer, sources = generate_answer(q)
    print('Top-3 hasil retrieval:')
    for s in sources:
        print(f"   {s['rank']}. ({s['similarity']}) [{s['category']}] {s['question']}")
    print('Jawaban:', answer)
    print('Sumber teratas:', sources[0]['question'] if sources else '-')
    print('---')
    time.sleep(13)  # jeda biar ga nabrak limit 5 request/menit
    test_log.append({
        'jenis': jenis,
        'query': q,
        'top1_question': sources[0]['question'],
        'top1_similarity': sources[0]['similarity'],
        'jawaban': answer.strip()
    })

[Ada di FAQ] Query: Bagaimana cara membatalkan pesanan?
Top-3 hasil retrieval:
   1. (0.788) [Pemesanan] Bagaimana cara membatalkan pesanan yang sudah dibuat?
   2. (0.7191) [Pengembalian] Bagaimana prosedur pengembalian barang yang tidak sesuai?
   3. (0.7079) [Pemesanan] Kenapa pesanan saya otomatis dibatalkan sistem?
Jawaban: Halo Kak! Untuk membatalkan pesanan, Kakak bisa melakukannya lewat halaman 'Pesanan Saya'. Pastikan status pesanan Kakak masih 'Diproses' ya. Kalau statusnya sudah 'Dikirim', pesanan tidak bisa dibatalkan.
Sumber teratas: Bagaimana cara membatalkan pesanan yang sudah dibuat?
---
[Ada di FAQ] Query: Berapa lama proses refund setelah barang dikembalikan?
Top-3 hasil retrieval:
   1. (0.8136) [Pengembalian] Berapa lama proses refund setelah pengembalian disetujui?
   2. (0.7318) [Pengembalian] Ke mana dana refund akan dikembalikan?
   3. (0.7205) [Pengembalian] Bagaimana prosedur pengembalian barang yang tidak sesuai?
Jawaban: Halo Kak! Untuk proses refund setelah

In [14]:
# rangkuman skor top-1 tiap query biar gampang dibandingin
pd.set_option('display.max_colwidth', 80)
pd.DataFrame(test_log)[['jenis', 'query', 'top1_question', 'top1_similarity']]

,jenis,query,top1_question,top1_similarity
0,Ada di FAQ,Bagaimana cara membatalkan pesanan?,Bagaimana cara membatalkan pesanan yang sudah dibuat?,0.7880
1,Ada di FAQ,Berapa lama proses refund setelah barang dikembalikan?,Berapa lama proses refund setelah pengembalian disetujui?,0.8136
2,Parafrase,"duit udah kepotong tapi orderan masih dibilang belum lunas, gimana nih?",Kenapa pembayaran saya berhasil dipotong tapi status pesanan masih belum ter...,0.7933
3,Parafrase,"aku ga inget kata sandi buat masuk, caranya reset gimana ya","Saya lupa password akun, bagaimana cara mengaturnya ulang?",0.7693
4,Parafrase,barang yang dateng beda sama yang aku order,"Saya menerima barang yang salah, apa yang harus saya lakukan?",0.8011
5,Tidak ada di FAQ,"Apakah bisa kirim barang ke luar negeri, misalnya ke Jepang?",Apakah bisa memilih jasa pengiriman sendiri saat checkout?,0.6932


### Catatan Analisis Relevansi

Total aku coba 6 query (2 yang ada di FAQ, 3 parafrase/bahasa santai, 1 yang ga ada di FAQ). Ringkasnya:

| # | Jenis | Query | Top-1 hasil retrieval | Similarity | Retrieval | Jawaban LLM |
|---|---|---|---|---|---|---|
| 1 | Ada di FAQ | Bagaimana cara membatalkan pesanan? | Bagaimana cara membatalkan pesanan yang sudah dibuat? | 0.788 | Tepat | Sesuai FAQ |
| 2 | Ada di FAQ | Berapa lama proses refund setelah barang dikembalikan? | Berapa lama proses refund setelah pengembalian disetujui? | 0.814 | Tepat | Sesuai FAQ (3-7 hari kerja) |
| 3 | Parafrase | duit udah kepotong tapi orderan masih dibilang belum lunas, gimana nih? | Kenapa pembayaran saya berhasil dipotong tapi status pesanan masih belum terbayar? | 0.793 | Tepat | Sesuai FAQ (tunggu 15-30 menit) |
| 4 | Parafrase | aku ga inget kata sandi buat masuk, caranya reset gimana ya | Saya lupa password akun, bagaimana cara mengaturnya ulang? | 0.769 | Tepat | Sesuai FAQ |
| 5 | Parafrase | barang yang dateng beda sama yang aku order | Saya menerima barang yang salah, apa yang harus saya lakukan? | 0.801 | Tepat | Sesuai, gabungin 2 FAQ |
| 6 | Tidak ada di FAQ | Apakah bisa kirim barang ke luar negeri, misalnya ke Jepang? | Apakah bisa memilih jasa pengiriman sendiri saat checkout? | 0.693 | Ga relevan (memang ga ada) | Ngaku ga tahu, arahkan ke CS |

**Query yang ada di FAQ (1-2)**
Dua-duanya top-1 nya langsung pas. Ini wajar karena kata-katanya hampir sama dengan pertanyaan di FAQ. Yang menarik di query 1, rank 2 nya malah "prosedur pengembalian barang" (kategori Pengembalian), padahal yang lebih nyambung harusnya "pesanan otomatis dibatalkan sistem" (rank 3). Kayaknya model nangkep "membatalkan" mirip sama "mengembalikan". Untungnya LLM cuma pakai FAQ rank 1, jadi jawabannya tetap benar.

**Query parafrase (3-5)**
Ini bagian yang paling aku pengen tes. Query 3 hampir ga ada kata yang sama dengan FAQ aslinya ("duit kepotong" vs "pembayaran dipotong", "belum lunas" vs "belum terbayar", "orderan" vs "pesanan") tapi tetap ketemu di rank 1 dengan skor 0.79. Query 4 juga sama, "ga inget kata sandi buat masuk" bisa nyambung ke "lupa password akun". Artinya semantic search memang nangkep makna, bukan sekadar nyocokin kata kunci. Kalau pakai keyword search biasa, query 3 dan 4 kemungkinan besar gagal.

Di query 5, top-3 nya semua dari kategori Pengembalian dan LLM ngegabungin info dari FAQ rank 1 (komplain "Barang Tidak Sesuai" + foto) dan rank 2 (maksimal 3 hari). Hasilnya jawaban jadi lebih lengkap dibanding cuma ambil satu FAQ. Ini salah satu keuntungan ngambil top-3, bukan top-1 doang.

**Query yang ga ada di FAQ (6)**
FAISS tetap ngembaliin 3 hasil (karena memang selalu ngasih top-k), isinya FAQ seputar pengiriman. Tapi ga ada satupun yang bahas pengiriman internasional. Bisa dilihat skornya juga paling rendah (0.693) dan ketiga hasilnya skornya mepet banget (0.693, 0.693, 0.689), beda sama query lain yang top-1 nya jelas lebih tinggi dari rank 2. Untungnya instruksi di prompt jalan: LLM ga ngarang dan bilang belum punya informasinya terus nyaranin hubungi CS.

**Kesimpulan & hal yang perlu diperhatiin**
- Retrieval akurat di 5/5 query yang memang ada jawabannya, termasuk yang parafrase.
- Skor similarity dari gemini-embedding-001 rentangnya sempit (kira-kira 0.65-0.82), jadi beda antara "relevan" dan "ga relevan" cuma sekitar 0.07-0.1. Dari percobaan ini, query yang relevan top-1 nya di atas 0.76, yang ga relevan di bawah 0.70. Kalau mau dikembangin, bisa pasang threshold (misal sekitar 0.72) supaya context yang ga nyambung ga usah dikirim ke LLM sama sekali.
- Penanganan "ga tahu" sekarang masih 100% bergantung sama LLM nurut ke prompt. Di tes ini berhasil, tapi kalau ada FAQ yang "mirip-mirip" bisa aja LLM maksa jawab pakai FAQ itu.
- Gaya bahasa LLM kadang manggil "Kak", kadang "kamu". Ga masalah buat prototype tapi kalau buat produksi perlu dipertegas di bagian Persona.
- Karena pakai temperature 0.3, kalimat jawaban bisa sedikit beda tiap kali di-run, tapi isinya tetap ngikutin FAQ.